In [1]:
from xaikd import models, datasets, utils, attributors

from xaikd.utils import metrics 

import pandas as pd
from matplotlib import pyplot as plt

import torch
from torch.utils.data import random_split

import numpy as np
from torch import nn

from tqdm.notebook import tqdm

from torch.nn import functional as F

In [2]:
DEVICE = utils.get_device()
DATASIZE = 0.1
SEED = 1

In [3]:
DF_SUPER_CLASSES = pd.read_csv("../../xaikd/resources/cifar100-label-mapping.csv")

In [4]:
DF_SUPER_CLASSES

,coarse_label_name,fine_label_name,fine_label,coarse_label
0,aquatic_mammals,beaver,4,0
1,food_containers,cup,28,3
2,aquatic_mammals,dolphin,30,0
3,small_mammals,hamster,36,16
4,large_natural_outdoor_scenes,plain,60,10
...,...,...,...,...
95,small_mammals,shrew,74,16
96,medium_mammals,raccoon,66,12
97,large_man_made_outdoor_things,road,68,9
98,aquatic_mammals,seal,72,0


In [5]:
model = models.get_trained_model("cifar100-resnet18-v1")
model.to(DEVICE);

In [6]:
dataset = datasets.construct("cifar100")

In [7]:

ds_train = dataset.create_subset(train_split=True)
ds_train
ds_val = dataset.create_subset(train_split=False)


def target_transform(x):
    row = DF_SUPER_CLASSES[DF_SUPER_CLASSES.fine_label == x]
    assert len(row) == 1, row.shape
    return row["coarse_label"].values[-1]
    
ds_train.target_transform = target_transform
ds_val.target_transform = target_transform

trng = torch.Generator()
trng.manual_seed(1)

ds_train, _ = random_split(ds_train, [DATASIZE, 1- DATASIZE], generator=trng)
ds_val, _ = random_split(ds_val, [DATASIZE, 1- DATASIZE], generator=trng)

dl_train = datasets.build_dataloader(ds_train, shuffle=False)
dl_val = datasets.build_dataloader(ds_val, shuffle=False)

In [8]:

class SuperclassModel(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
    def forward(self, x):
        return self.superclass_prob(x)
        
    def superclass_prob(self, x):
        logits = self.model(x)
        prob = torch.softmax(logits, dim=1)
    
        b,  _ = prob.shape
        num_superclasses = 20
        
        prob_superclasses = torch.zeros((b, num_superclasses))
        for superclass_ix in range(num_superclasses):
            fine_labels = DF_SUPER_CLASSES[DF_SUPER_CLASSES.coarse_label == superclass_ix].fine_label.values
            prob_superclasses[:, superclass_ix] = prob[:, fine_labels].sum(dim=1)
        return prob_superclasses


ref_acc, ref_xnt = metrics.accuracy(
    SuperclassModel(model),
    dl_val,
    num_classes=20,
    device=DEVICE
)
ref_acc, ref_xnt

(0.8489999771118164, 2.2732491493225098)

In [14]:
def _solve_eigvecs(cov, sort_func=lambda x: x):
    eigvals, eigvecs = np.linalg.eigh(cov)

    assert len(eigvals.shape) == 1

    indices = np.argsort(-sort_func(eigvals))
    eigvals = eigvals[indices]
    eigvecs = eigvecs[:, indices]

    return eigvecs

def estimate_basis(basis_name, arr_act, arr_ctx):
    if basis_name == "pca":
        cov = arr_act.T @ arr_act
        eigvecs = _solve_eigvecs(cov)
        return eigvecs
    elif basis_name == "gpca":
        cov = arr_ctx.T @ arr_ctx
        eigvecs = _solve_eigvecs(cov)
        return eigvecs
    elif basis_name == "v0":
        w = model.fc.weight[0, :].detach().cpu().numpy()
        w = w / np.linalg.norm(w)
        U = w.reshape((-1, 1))
        return U
    # elif basis_name == "prca":
    #     A = arr_act 
    #     C = arr_ctx 
        
    #     cov = A.T @ C + C.T @ A
    #     eigvecs = _solve_eigvecs(cov)
    #     return eigvecs
    elif basis_name == "prca-sortabs":
        cov = arr_act.T @ arr_ctx + arr_ctx.T @ arr_act
        eigvecs = _solve_eigvecs(cov, sort_func=np.abs)
        return eigvecs
    elif basis_name == "prca-signed":
        arr_rel = (arr_act * arr_ctx).sum(axis=1, keepdims=True)
        arr_ctx_signed = (arr_rel >= 0) * arr_ctx - (arr_rel < 0) * arr_ctx
        return estimate_basis("prca-sortabs", arr_act, arr_ctx_signed)
    else:
        raise

In [15]:
np.arange(5).tolist().index(2)

2

In [16]:
def construct_fh(Uk):
    def fh(mod, inp, outp):
        return F.conv2d(
            outp,
            (Uk@Uk.T).unsqueeze(2).unsqueeze(3)
        )
    return fh


class WinningClassEvidenceSquared(attributors.LogitModifier):
    def __init__(self, num_classes: int) -> None:
        self.num_classes = num_classes

    def __call__(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        logits = logits.clone()
        wining_targets = torch.argmax(logits, dim=1)
        return logits * F.one_hot(wining_targets, self.num_classes).to(logits.device)

    def __str__(self) -> str:
        return "winingclass"


class LikelihoodRatio(attributors.LogitModifier):
    def __init__(self, num_classes: int) -> None:
        self.num_classes = num_classes

    def __call__(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        logits = logits.clone()
        b, _ = logits.shape
        wining_targets = torch.argmax(logits, dim=1)

        grad_likelihood_ratio = torch.zeros(b, 1)
        for i in range(b):
            y = float(wining_targets[i].detach().cpu().numpy())

            coarse_label = target_transform(y)
            # print(f"[i={i}] y={y} ({coarse_label})")
            fine_labels = DF_SUPER_CLASSES[DF_SUPER_CLASSES.coarse_label == coarse_label].fine_label.values.tolist()
            probs = torch.softmax(logits[i, fine_labels], dim=0)
            grad_likelihood_ratio[i, 0] = probs[fine_labels.index(y)]
        return logits * (
            F.one_hot(wining_targets, self.num_classes).to(logits.device) * grad_likelihood_ratio.to(logits.device)
        )


class AllClassEvidence(attributors.LogitModifier):
    def __init__(self, num_classes: int) -> None:
        self.num_classes = num_classes

    def __call__(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        return logits.abs()

def compute_task_performance_at_k(
    model, layer, arr_ks,
    arr_basis_names=[
        "pca", 
        "prca-sortabs",
        "prca-signed"
    ]
):
    # todo: we need to use something else
    # logit_mod = attributors.WinningClassEvidence(num_classes=100)
    logit_mod = WinningClassEvidenceSquared(num_classes=100)
    # logit_mod = LikelihoodRatio(num_classes=100)
    # logit_mod = AllClassEvidence(num_classes=100)
    rng = np.random.default_rng(seed=1)
    arr_act, arr_ctx = attributors.extract_activation_context(
        model=model, 
        layer=layer, 
        dataset=dataset,
        data_loader=dl_train,
        logit_modifier=logit_mod,
        rng=rng,
        device=DEVICE,
    )
    arr_stat_rows = []
    
    module = getattr(model, layer)
    
    for basis_name in arr_basis_names:
        U = estimate_basis(basis_name, arr_act, arr_ctx)
        for k in tqdm(arr_ks, desc=f"[basis={basis_name}] compute task performance at k"):
            Uk = torch.from_numpy(U[:, :k]).to(DEVICE)
            row = dict(
                layer=layer,
                basis_name=basis_name,
                k=k,
            )
            try:
                hook = module.register_forward_hook(construct_fh(Uk))
                
                for label, dl in [
                    # ("train", dl_train_noshuffle),
                    ("val", dl_val)
                ]:
                    row[f"{label}_acc"] = metrics.accuracy(
                        SuperclassModel(model),
                        dl,
                        num_classes=20,
                        device=DEVICE,
                        verbose=False,
                    )[0]
                
            finally:
                hook.remove()
            arr_stat_rows.append(row)
            
    df = pd.DataFrame(arr_stat_rows)
    
    return df
compute_task_performance_at_k(model, "layer2", np.arange(1, 20, 2).tolist() + [64])

100%|██████████████████████████████████████████████████████████| 79/79 [00:06<00:00, 12.90it/s]


layer2: output-dims=torch.Size([128, 16, 16])


[basis=pca] compute task performance at k:   0%|          | 0/11 [00:00<?, ?it/s]

[basis=prca-sortabs] compute task performance at k:   0%|          | 0/11 [00:00<?, ?it/s]

[basis=prca-signed] compute task performance at k:   0%|          | 0/11 [00:00<?, ?it/s]

,layer,basis_name,k,val_acc
0,layer2,pca,1,0.041
1,layer2,pca,3,0.043
2,layer2,pca,5,0.120
3,layer2,pca,7,0.208
4,layer2,pca,9,0.324
5,layer2,pca,11,0.474
6,layer2,pca,13,0.612
7,layer2,pca,15,0.678
8,layer2,pca,17,0.717
9,layer2,pca,19,0.765


In [ ]:
def getting_performance(arr_layers):
    arr_layer_dims = utils.get_dimensions_at_layers(
        model=model,
        dataloader=dl_train,
        layers=arr_layers,
        device=DEVICE
    )

    arr_dfs = []

    for layer in tqdm(arr_layers, desc="getting results for layer"):
        d = arr_layer_dims[layer]
        # if layer == "layer4":
        last_fine_d = 50
        # else:
        #     last_fine_d = 20
        arr_ks = sorted(set(
            np.arange(1, last_fine_d).tolist() + 
            np.linspace(last_fine_d + 1, d, 8).astype(int).tolist()
        ))
        df = compute_task_performance_at_k(
            model, 
            layer=layer, 
            arr_ks=arr_ks,
            arr_basis_names=["pca", "prca-sortabs", "prca-signed"]
        )
        
        arr_dfs.append(df)

    return pd.concat(arr_dfs)

df = getting_performance(
    arr_layers=["layer1", "layer2", "layer3", "layer4"],
)

getting results for layer:   0%|          | 0/4 [00:00<?, ?it/s]


100%|██████████████████████████████████████████████████████████| 79/79 [00:05<00:00, 13.59it/s]


layer1: output-dims=torch.Size([64, 32, 32])


[basis=pca] compute task performance at k:   0%|          | 0/57 [00:00<?, ?it/s]

[basis=prca-sortabs] compute task performance at k:   0%|          | 0/57 [00:00<?, ?it/s]

...


100%|██████████████████████████████████████████████████████████| 79/79 [00:05<00:00, 14.03it/s]


layer4: output-dims=torch.Size([512, 4, 4])


[basis=pca] compute task performance at k:   0%|          | 0/57 [00:00<?, ?it/s]

[basis=prca-sortabs] compute task performance at k:   0%|          | 0/57 [00:00<?, ?it/s]

[basis=prca-signed] compute task performance at k:   0%|          | 0/57 [00:00<?, ?it/s]

In [ ]:
def viz_results(df):


    arr_layers = df.layer.unique()
    ncols = len(arr_layers)
    
    plt.figure(figsize=(3*ncols, 2))
    plt.suptitle(f"cifar100-superclasses (data_size={DATASIZE})", y=1.1)
    metric_col_name = "val_acc"
    for lix, layer in enumerate(arr_layers):
        plt.subplot(1, ncols, lix + 1)
        plt.title(f"Layer {layer}")

        for bix, basis_name in enumerate(["pca", "prca-sortabs", "prca-signed"]):
            
            _df = df[
             (df.basis_name == basis_name) & (df.layer == layer)
            ]
            if bix == 0:
                plt.axhline(
                    _df[metric_col_name].values[-1],
                    ls="--",
                    lw=1, 
                    color="k"
                )
            plt.plot(
                _df.k,
                _df[metric_col_name],
                label=basis_name
            )

            d = _df.k.values[-1]
            
        plt.ylim([0.5, 1.0])
        if lix == 0:
            plt.ylabel("Superclass ACC (val)")
            plt.legend()
        plt.xlabel(f"Subspace Dimensions K\n(d={d})")

viz_results(df)

# Dev
## 2024/12/18
- write superclass prediction head output 20 probs